In [ ]:
#!pip install mamba-ssm causal-conv1d --no-build-isolation

In [ ]:
import os, torch
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)


In [ ]:
import random
import numpy as np
import torch.nn as nn
import torch.optim as optim

GLOBAL_SEED = 122122

def set_seed(seed=GLOBAL_SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # Applies if you run on multi-GPU nodes

    print(f"Random seed is set to {seed}!")

set_seed(GLOBAL_SEED)

In [ ]:
from mamba_ssm import Mamba
import time

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def get_batch(X_data, Y_data, batch_size):
    for i in range(0, len(X_data), batch_size):
        X_batch = X_data[i:i + batch_size]
        Y_batch = Y_data[i:i + batch_size]

        if not X_batch:
            continue

        max_len = max([x.shape[1] for x in X_batch])

        padded_X_batch = []
        for x in X_batch:
            padding_needed = max_len - x.shape[1]
            padded_x = torch.nn.functional.pad(x, (0, padding_needed), 'constant', 0)
            padded_X_batch.append(padded_x)

        yield torch.cat(padded_X_batch, dim=0), Y_batch


num_epochs = 10
batch_size = 32

# Move model to device if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

In [ ]:
import pandas as pd

In [ ]:
csv_files = [
    'mcem-stock.csv',
    'tgls-stock.csv',
    'ttam-stock.csv',
    'uslm-stock.csv',
    'knf-stock.csv',
    'exp-stock.csv',
    'jhx-stock.csv',
    'mlm-stock.csv',
    'vmc-stock.csv',
    'crh-stock.csv'
]

# Mapping from filename part to ticker symbol in NG4SQ2NZ.csv
# This mapping is crucial to link news data to correct stock price columns
ticker_mapping = {
    'mcem': 'MCEM',
    'tgls': 'TGLS',
    'ttam': 'TTAM',
    'uslm': 'USLM',
    'knf': 'KNF',
    'exp': 'EXP',
    'jhx': 'JHX',
    'mlm': 'MLM',
    'vmc': 'VMC',
    'crh': 'CRH'
}

all_news_dfs = []

for file in csv_files:
    temp_df = pd.read_csv(file)
    # Extract ticker from filename
    filename_prefix = file.replace('-stock.csv', '')

    # Use the mapping to get the correct ticker symbol for merging
    ticker_symbol = ticker_mapping.get(filename_prefix, None)

    if ticker_symbol:
        temp_df['Ticker'] = ticker_symbol
        all_news_dfs.append(temp_df)
    else:
        print(f"Warning: No ticker mapping found for {file}. Skipping.")

# Concatenate all individual news dataframes into a single DataFrame
df = pd.concat(all_news_dfs, ignore_index=True)

# Ensure 'datetime' in df is in datetime format and then extract just the date
df['Date'] = pd.to_datetime(df['datetime'], format='%m/%d/%Y %I:%M:%S %p').dt.normalize()

# Encode the titles after the DataFrame has been fully constructed
'''encoded = tokenizer(
    df['title'].tolist(),
    padding=True,
    truncation=True,
    max_length=64, # для заголовков обычно достаточно
    return_tensors='pt'
)'''

# 1. Сохраняем в DataFrame (если нужно для наглядности)
# Мы переводим тензор в список списков, чтобы pandas их "съел"
#df['encoded_title'] = encoded['input_ids'].tolist()
df['encoded_title'] = df['title'].apply(lambda x: tokenizer.encode(x, add_special_tokens=True, truncation=True, padding=True, return_tensors='pt'))
df.head()

In [ ]:
# Load the stock data
stock_df_raw = pd.read_csv('Tickers.csv')

def process_single_stock_data(raw_stock_df, ticker_column_name):
    stock_data = raw_stock_df[['Ticker', ticker_column_name]].copy()

    stock_data.rename(columns={'Ticker': 'Date', ticker_column_name: 'Close'}, inplace=True)
    stock_data = stock_data.iloc[2:].copy()
    stock_data['Date'] = pd.to_datetime(stock_data['Date'])
    stock_data['Close'] = pd.to_numeric(stock_data['Close'], errors='coerce')
    stock_data.dropna(subset=['Close'], inplace=True)
    stock_data.sort_values(by='Date', inplace=True)
    stock_data.reset_index(drop=True, inplace=True)

    # Calculate the daily percentage change (Y)
    stock_data['Percentage_Change'] = ((stock_data['Close'].shift(-1) - stock_data['Close']) / stock_data['Close'].shift(-1)) * 100

    # Drop the last row as it will have NaN for 'Percentage_Change'
    stock_data.dropna(subset=['Percentage_Change'], inplace=True)

    return stock_data

print("Stock data processing function defined.")

In [ ]:
# Ensure 'Date' column is already prepared in 'df' (done in previous cell)

# Get a list of unique tickers present in the combined news DataFrame
unique_tickers_in_news = df['Ticker'].unique()

all_combined_dfs = []

for ticker in unique_tickers_in_news:
    print(f"Processing data for ticker: {ticker}")

    # Filter news for the current ticker
    news_for_ticker = df[df['Ticker'] == ticker].copy()

    # Process stock data for the current ticker
    # We need to make sure the column name in stock_df_raw matches the ticker
    # For 'JHX', the stock_df_raw column is also 'JHX'
    stock_data_for_ticker = process_single_stock_data(stock_df_raw, ticker)

    # Merge the news DataFrame with the processed stock DataFrame on 'Date'
    # We will use an inner merge to only keep dates where both news and stock data are available.
    merged_df_for_current_ticker = pd.merge(news_for_ticker, stock_data_for_ticker, on='Date', how='inner')

    all_combined_dfs.append(merged_df_for_current_ticker)

# Concatenate all ticker-specific merged DataFrames into a single combined_df
combined_df = pd.concat(all_combined_dfs, ignore_index=True)

# Sort by date to ensure chronological order for time series data
combined_df.sort_values(by='Date', inplace=True)
combined_df.reset_index(drop=True, inplace=True)

print("Combined DataFrame Head (News and Stock Data):")
display(combined_df.head())
print("\nCombined DataFrame Info:")
combined_df.info()

# Define X (encoded titles) and Y (percentage change)
X = combined_df['encoded_title']
Y = torch.tensor(combined_df['Percentage_Change'].values, dtype=torch.float32).unsqueeze(1) # Ensure Y is a column vector

print(f"Shape of X (encoded_title): {X.shape}")
print(f"Shape of Y (Percentage_Change): {Y.shape}")

In [ ]:
X_list = X.tolist()

split_ratio = 0.8
split_index = int(len(combined_df) * split_ratio)

X_train_list = X_list[:split_index]
Y_train = Y[:split_index]

X_val_list = X_list[split_index:]
Y_val = Y[split_index:]

print(f"training samples: {len(X_train_list)}")
print(f"validation samples: {len(X_val_list)}")

In [ ]:
vocab_size = tokenizer.vocab_size
output_dim=1
best_params = {'embedding_dim': 50, 'hidden_dim': 128, 'lr': 1e-05}

### RNN Model Definition

First, let's define the RNN model architecture.

In [ ]:
set_seed(GLOBAL_SEED)
class RNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(RNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):

        embedded = self.embedding(text)

        output, hidden = self.rnn(embedded)
        hidden = hidden.squeeze(0)

        output = self.fc(hidden)

        return output

Best parameters block

In [ ]:
import itertools

# Define the parameter grid for GridSearch
param_grid_rnn = {
    'embedding_dim': [50, 100],
    'hidden_dim': [64, 128, 256],
    'lr': [1e-4, 1e-5, 1e-6]
}

best_val_loss = float('inf')
best_params = {}

print("Starting RNN GridSearch...")

# Iterate through all combinations of parameters
for embedding_dim, hidden_dim, lr in itertools.product(
    param_grid_rnn['embedding_dim'],
    param_grid_rnn['hidden_dim'],
    param_grid_rnn['lr']
):
    print(f"\nTraining RNN with: embedding_dim={embedding_dim}, hidden_dim={hidden_dim}, lr={lr}")

    set_seed(GLOBAL_SEED)

    model_rnn = RNN(
        vocab_size,
        embedding_dim,
        hidden_dim,
        output_dim
    ).to(device)

    optimizer_rnn = optim.AdamW(model_rnn.parameters(), lr=lr)
    criterion_rnn = nn.MSELoss()

    # Temporary lists to store losses for the current parameter combination
    current_train_losses = []
    current_val_losses = []

    for epoch in range(num_epochs):
        model_rnn.train()
        total_loss = 0
        for X_batch, Y_batch in get_batch(X_train_list, Y_train, batch_size):
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            optimizer_rnn.zero_grad()
            predictions = model_rnn(X_batch)
            loss = criterion_rnn(predictions, Y_batch)
            loss.backward()
            optimizer_rnn.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / (len(X_train_list) / batch_size)
        current_train_losses.append(avg_train_loss)

        model_rnn.eval()
        val_total_loss = 0
        with torch.no_grad():
            for X_val_batch, Y_val_batch in get_batch(X_val_list, Y_val, batch_size):
                X_val_batch = X_val_batch.to(device)
                Y_val_batch = Y_val_batch.to(device)

                val_predictions = model_rnn(X_val_batch)
                val_loss = criterion_rnn(val_predictions, Y_val_batch)
                val_total_loss += val_loss.item()

        avg_val_loss = val_total_loss / (len(X_val_list) / batch_size)
        current_val_losses.append(avg_val_loss)

        # print(f"  Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    # After all epochs for the current combination, check if it's the best
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_params = {
            'embedding_dim': embedding_dim,
            'hidden_dim': hidden_dim,
            'lr': lr,
            'val_loss': best_val_loss
        }

print("\nRNN GridSearch complete!")
print(f"Best RNN parameters found: {best_params}")

In [ ]:
best_params = {'embedding_dim': 50, 'hidden_dim': 128, 'lr': 1e-05}

Back to the model

In [ ]:
fixed_embedding_dim_for_rnn = best_params['embedding_dim']
fixed_hidden_dim_for_rnn = best_params['hidden_dim']
fixed_lr_for_rnn = best_params['lr']

print(f"Training RNN with: embedding_dim={fixed_embedding_dim_for_rnn}, hidden_dim={fixed_hidden_dim_for_rnn}, lr={fixed_lr_for_rnn}")


set_seed(GLOBAL_SEED)
model_rnn_comparison = RNN(
    vocab_size,
    fixed_embedding_dim_for_rnn,
    fixed_hidden_dim_for_rnn,
    output_dim
).to(device)

optimizer_rnn_comparison = optim.AdamW(model_rnn_comparison.parameters(), lr=fixed_lr_for_rnn)
criterion_rnn_comparison = nn.MSELoss()

comparison_train_losses = []
comparison_val_losses = []

start_time = time.perf_counter()

for epoch in range(num_epochs):
    model_rnn_comparison.train()
    total_loss = 0
    for X_batch, Y_batch in get_batch(X_train_list, Y_train, batch_size):
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer_rnn_comparison.zero_grad()
        predictions = model_rnn_comparison(X_batch)
        loss = criterion_rnn_comparison(predictions, Y_batch)
        loss.backward()
        optimizer_rnn_comparison.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / (len(X_train_list) / batch_size)
    comparison_train_losses.append(avg_train_loss)

    model_rnn_comparison.eval()
    val_total_loss = 0
    with torch.no_grad():
        for X_val_batch, Y_val_batch in get_batch(X_val_list, Y_val, batch_size):
            X_val_batch = X_val_batch.to(device)
            Y_val_batch = Y_val_batch.to(device)

            val_predictions = model_rnn_comparison(X_val_batch)
            val_loss = criterion_rnn_comparison(val_predictions, Y_val_batch)
            val_total_loss += val_loss.item()

    avg_val_loss = val_total_loss / (len(X_val_list) / batch_size)
    comparison_val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

print("RNN comparison model training complete!")

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Function took {execution_time:.6f} seconds to run.")

In [ ]:
# Plotting the training and validation loss for the comparison LSTM model
epochs = range(1, len(comparison_train_losses) + 1)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Training Loss on the primary y-axis (ax1)
ax1.plot(epochs, comparison_train_losses, 'bo-', label='Training loss (RNN)')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Training Loss', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a second y-axis for Validation Loss (ax2)
ax2 = ax1.twinx()
ax2.plot(epochs, comparison_val_losses, 'ro-', label='Validation loss (RNN)')
ax2.set_ylabel('Validation Loss', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Add title and legend
plt.title('Training and Validation Loss (RNN)')
fig.legend(loc='upper right', bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)

plt.grid(True)
plt.savefig('training_validation_loss_RNN_comparison.png')
plt.show()

In [ ]:
#total_params_optimized = sum(p.numel() for p in model_rnn_comparison.parameters() if p.requires_grad)
print(f"Total parameters in the best optimized model: {total_params_optimized:,}")

### LSTM Model Definition

First, let's define the LSTM model architecture. Similar to the RNN, it will have an Embedding layer, but will use an `nn.LSTM` layer instead of `nn.RNN`.

In [ ]:
set_seed(GLOBAL_SEED)
class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(LSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):

        embedded = self.embedding(text)

        output, (hidden, cell) = self.lstm(embedded)
        output = self.fc(hidden)

        return output

print("LSTM Model class defined.")

#### Instantiate LSTM Model for Comparison (with RNN's best parameters)

### LSTM Model with RNN's Best Hyperparameters for Direct Comparison

To allow for a more direct comparison with the RNN model, we will train an LSTM model using the best `embedding_dim`, `hidden_dim`, and `lr` found for the RNN, and fix `num_layers=1` and `bidirectional=False` for the LSTM.

In [ ]:
fixed_embedding_dim_for_lstm = best_params['embedding_dim']
fixed_hidden_dim_for_lstm = best_params['hidden_dim']
fixed_lr_for_lstm = best_params['lr']

print(f"Training LSTM with: embedding_dim={fixed_embedding_dim_for_lstm}, hidden_dim={fixed_hidden_dim_for_lstm}, lr={fixed_lr_for_lstm}")
set_seed(GLOBAL_SEED)

model_lstm_comparison = LSTM(
    vocab_size,
    fixed_embedding_dim_for_lstm,
    fixed_hidden_dim_for_lstm,
    output_dim
).to(device)

optimizer_lstm_comparison = optim.AdamW(model_lstm_comparison.parameters(), lr=fixed_lr_for_lstm)
criterion_lstm_comparison = nn.MSELoss()

comparison_train_losses = []
comparison_val_losses = []

start_time = time.perf_counter()

for epoch in range(num_epochs):
    model_lstm_comparison.train()
    total_loss = 0
    for X_batch, Y_batch in get_batch(X_train_list, Y_train, batch_size):
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer_lstm_comparison.zero_grad()
        predictions = model_lstm_comparison(X_batch)
        loss = criterion_lstm_comparison(predictions, Y_batch)
        loss.backward()
        optimizer_lstm_comparison.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / (len(X_train_list) / batch_size)
    comparison_train_losses.append(avg_train_loss)

    model_lstm_comparison.eval()
    val_total_loss = 0
    with torch.no_grad():
        for X_val_batch, Y_val_batch in get_batch(X_val_list, Y_val, batch_size):
            X_val_batch = X_val_batch.to(device)
            Y_val_batch = Y_val_batch.to(device)

            val_predictions = model_lstm_comparison(X_val_batch)
            val_loss = criterion_lstm_comparison(val_predictions, Y_val_batch)
            val_total_loss += val_loss.item()

    avg_val_loss = val_total_loss / (len(X_val_list) / batch_size)
    comparison_val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

print("LSTM comparison model training complete!")

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Function took {execution_time:.6f} seconds to run.")

In [ ]:
# Plotting the training and validation loss for the comparison LSTM model
epochs = range(1, len(comparison_train_losses) + 1)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Training Loss on the primary y-axis (ax1)
ax1.plot(epochs, comparison_train_losses, 'bo-', label='Training loss (LSTM)')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Training Loss', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a second y-axis for Validation Loss (ax2)
ax2 = ax1.twinx()
ax2.plot(epochs, comparison_val_losses, 'ro-', label='Validation loss (LSTM)')
ax2.set_ylabel('Validation Loss', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Add title and legend
plt.title('Training and Validation Loss (LSTM)')
fig.legend(loc='upper right', bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)

plt.grid(True)
plt.savefig('training_validation_loss_lstm_comparison.png')
plt.show()

### GRU Model with RNN's Best Hyperparameters for Direct Comparison

To allow for a more direct comparison with the RNN model, we will train an GRU model using the best `embedding_dim`, `hidden_dim`, and `lr` found for the RNN, and fix `num_layers=1` and `bidirectional=False` for the GRU.

In [ ]:
set_seed(GLOBAL_SEED)
class GRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(GRU, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        output, hidden = self.gru(embedded)
        output = self.fc(hidden)

        return output

print("GRU Model class defined.")

In [ ]:
fixed_embedding_dim_for_gru = best_params['embedding_dim']
fixed_hidden_dim_for_gru= best_params['hidden_dim']
fixed_lr_for_gru = best_params['lr']


print(f"Training GRU with: embedding_dim={fixed_embedding_dim_for_gru}, hidden_dim={fixed_hidden_dim_for_gru}, lr={fixed_lr_for_gru}")
set_seed(GLOBAL_SEED)
model_gru_comparison = GRU(
    vocab_size,
    fixed_embedding_dim_for_gru,
    fixed_hidden_dim_for_gru,
    output_dim
).to(device)

optimizer_gru_comparison = optim.AdamW(model_gru_comparison.parameters(), lr=fixed_lr_for_gru)
criterion_gru_comparison = nn.MSELoss()

comparison_train_losses = []
comparison_val_losses = []

start_time = time.perf_counter()

for epoch in range(num_epochs):
    model_gru_comparison.train()
    total_loss = 0
    for X_batch, Y_batch in get_batch(X_train_list, Y_train, batch_size):
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer_gru_comparison.zero_grad()
        predictions = model_gru_comparison(X_batch)
        loss = criterion_gru_comparison(predictions, Y_batch)
        loss.backward()
        optimizer_gru_comparison.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / (len(X_train_list) / batch_size)
    comparison_train_losses.append(avg_train_loss)

    model_gru_comparison.eval()
    val_total_loss = 0
    with torch.no_grad():
        for X_val_batch, Y_val_batch in get_batch(X_val_list, Y_val, batch_size):
            X_val_batch = X_val_batch.to(device)
            Y_val_batch = Y_val_batch.to(device)

            val_predictions = model_gru_comparison(X_val_batch)
            val_loss = criterion_gru_comparison(val_predictions, Y_val_batch)
            val_total_loss += val_loss.item()

    avg_val_loss = val_total_loss / (len(X_val_list) / batch_size)
    comparison_val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

print("GRU comparison model training complete!")

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Function took {execution_time:.6f} seconds to run.")

In [ ]:
# Plotting the training and validation loss for the comparison GRU model
epochs = range(1, len(comparison_train_losses) + 1)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Training Loss on the primary y-axis (ax1)
ax1.plot(epochs, comparison_train_losses, 'bo-', label='Training loss (GRU)')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Training Loss', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a second y-axis for Validation Loss (ax2)
ax2 = ax1.twinx()
ax2.plot(epochs, comparison_val_losses, 'ro-', label='Validation loss (GRU)')
ax2.set_ylabel('Validation Loss', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Add title and legend
plt.title('Training and Validation Loss (GRU)')
fig.legend(loc='upper right', bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)

plt.grid(True)
plt.savefig('training_validation_loss_GRU_comparison.png')
plt.show()

### Mamba Model with RNN's Best Hyperparameters for Direct Comparison

To allow for a more direct comparison with the RNN model, we will train an Mamba model using the best `embedding_dim`, `hidden_dim`, and `lr` found for the RNN, and fix `num_layers=1` and `bidirectional=False` for the Mamba.

### Training Loop for Mamba Model

In [ ]:
set_seed(GLOBAL_SEED)
class MambaModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim):
        super(MambaModel, self).__init__()
        # 1. Map token IDs to vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 2. Complete Mamba block
        self.mamba = Mamba(
            d_model=embedding_dim,  # Model dimension
            d_state=16,            # SSM state dimension (default: 16)
            d_conv=4,              # Local convolution width (default: 4)
            expand=2               # Block expansion factor (default: 2)
        )

        # 3. Final classification head
        self.fc = nn.Linear(embedding_dim, output_dim)

    def forward(self, text):
        # input shape: [batch_size, seq_len]
        embedded = self.embedding(text)

        # Mamba expects shape: [batch_size, seq_len, embedding_dim]
        # It returns a tensor of the exact same shape
        mamba_out = self.mamba(embedded)

        # Pooling: extract the representation of the last token
        last_token = mamba_out[:, -1, :]

        # Final projection to output dimensions
        output = self.fc(last_token)
        return output

print("Mamba Model class defined.")

In [ ]:
fixed_embedding_dim_for_mamba = best_params['embedding_dim']
fixed_lr_for_mamba = best_params['lr']

print(f"Training Mamba with: embedding_dim={fixed_embedding_dim_for_mamba}, lr={fixed_lr_for_mamba}")

# 2. Initialize the Mamba model (Ensure device is set to 'cuda')
set_seed(GLOBAL_SEED)
model_mamba_comparison = MambaModel(
    vocab_size=vocab_size,
    embedding_dim=fixed_embedding_dim_for_mamba,
    output_dim=output_dim
).to(device)

# 3. Setup optimizer and loss function (MSELoss for regression)
optimizer_mamba_comparison = optim.AdamW(model_mamba_comparison.parameters(), lr=fixed_lr_for_mamba)
criterion_mamba_comparison = nn.MSELoss()

comparison_train_losses_mamba = []
comparison_val_losses_mamba = []

start_time = time.perf_counter()

# 4. Training Loop
for epoch in range(num_epochs):
    model_mamba_comparison.train()
    total_loss = 0
    for X_batch, Y_batch in get_batch(X_train_list, Y_train, batch_size):
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer_mamba_comparison.zero_grad()

        # Mamba kernels expect mixed precision (Bfloat16/Float16) to run correctly
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            predictions = model_mamba_comparison(X_batch)
            # Ensure predictions match Y_batch type (Float32) for MSE loss calculation
            loss = criterion_mamba_comparison(predictions.float(), Y_batch.float())

        loss.backward()
        optimizer_mamba_comparison.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / (len(X_train_list) / batch_size)
    comparison_train_losses_mamba.append(avg_train_loss)

    # 5. Validation Loop
    model_mamba_comparison.eval()
    val_total_loss = 0
    with torch.no_grad():
        for X_val_batch, Y_val_batch in get_batch(X_val_list, Y_val, batch_size):
            X_val_batch = X_val_batch.to(device)
            Y_val_batch = Y_val_batch.to(device)

            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                val_predictions = model_mamba_comparison(X_val_batch)
                val_loss = criterion_mamba_comparison(val_predictions.float(), Y_val_batch.float())

            val_total_loss += val_loss.item()

    avg_val_loss = val_total_loss / (len(X_val_list) / batch_size)
    comparison_val_losses_mamba.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

print("Mamba comparison model training complete!")

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Function took {execution_time:.6f} seconds to run.")

In [ ]:
params_mamba = sum(p.numel() for p in model_mamba_comparison.parameters() if p.requires_grad)
print(f"Total Mamba Parameters: {params_mamba:,}")

### Plotting Training and Validation Loss for Mamba Model

In [ ]:
# Plotting the training and validation loss for the Mamba model
epochs = range(1, len(comparison_train_losses_mamba) + 1)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Training Loss on the primary y-axis (ax1)
ax1.plot(epochs, comparison_train_losses_mamba, 'bo-', label='Training loss (Mamba)')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Training Loss', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a second y-axis for Validation Loss (ax2)
ax2 = ax1.twinx()
ax2.plot(epochs, comparison_val_losses_mamba, 'ro-', label='Validation loss (Mamba)')
ax2.set_ylabel('Validation Loss', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Add title and legend
plt.title('Training and Validation Loss (Mamba)')
fig.legend(loc='upper right', bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)

plt.grid(True)
plt.savefig('training_validation_loss_mamba.png')
plt.show()

In [ ]:
# Загружаем оригинальный FinBERT
finbert_name = "ProsusAI/finbert"
orig_model = AutoModel.from_pretrained(finbert_name)

# Извлекаем веса эмбеддингов (это тензор)
finbert_embeddings_weights = orig_model.embeddings.word_embeddings.weight.data.clone()

In [ ]:
set_seed(GLOBAL_SEED)
class FinRegressionModel(nn.Module):
    def __init__(self, emb_weights, hidden_dim=128, num_layers=8):
        super().__init__()

        # 1. Frozen FinBERT embeddings
        self.embedding = nn.Embedding.from_pretrained(emb_weights, freeze=True)

        # 2. Projection layer
        self.proj = nn.Linear(768, hidden_dim)

        # 3. Learnable positional embedding (Fixed max length, e.g., 512)
        self.pos_embedding = nn.Parameter(torch.zeros(1, 512, hidden_dim))

        # 4. Transformer Encoder blocks
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=4,
            dim_feedforward=hidden_dim * 4,
            batch_first=True,
            dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 5. Output regression head
        self.regressor = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        # input_ids shape: [batch_size, seq_len]
        x = self.embedding(input_ids)
        x = self.proj(x)

        # Slice positional embedding to match current batch's sequence length
        x = x + self.pos_embedding[:, :input_ids.size(1), :]

        # Forward pass through the Transformer blocks without masking
        x = self.encoder(x)

        # Pooling: Extract the first token (CLS token equivalent) for regression
        return self.regressor(x[:, 0, :])

# Пример проверки параметров:
model = FinRegressionModel(finbert_embeddings_weights)
print(f"Всего параметров: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# 2. Инициализируем модель BERT-like
model_bert_comparison = FinRegressionModel(finbert_embeddings_weights).to(device)

# 3. Оптимизатор и лосс (те же, что для GRU)
set_seed(GLOBAL_SEED)
optimizer_bert_comparison = optim.AdamW(model_bert_comparison.parameters(), lr=fixed_lr_for_gru)
criterion_bert_comparison = nn.MSELoss()

comparison_bert_train_losses = []
comparison_bert_val_losses = []

start_time = time.perf_counter()

for epoch in range(num_epochs):
    model_bert_comparison.train()
    total_loss = 0
    for X_batch, Y_batch in get_batch(X_train_list, Y_train, batch_size):
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device).view(-1, 1) # Убеждаемся, что Y имеет размер (batch, 1)

        optimizer_bert_comparison.zero_grad()

        # ВАЖНО: BERT ожидает (batch, seq_len)
        predictions = model_bert_comparison(X_batch)

        loss = criterion_bert_comparison(predictions, Y_batch)
        loss.backward()

        # Полезно для BERT: ограничение градиентов (clipping), чтобы модель не "разлеталась"
        torch.nn.utils.clip_grad_norm_(model_bert_comparison.parameters(), max_norm=1.0)

        optimizer_bert_comparison.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / (len(X_train_list) / batch_size)
    comparison_bert_train_losses.append(avg_train_loss)

    # Валидация
    model_bert_comparison.eval()
    val_total_loss = 0
    with torch.no_grad():
        for X_val_batch, Y_val_batch in get_batch(X_val_list, Y_val, batch_size):
            X_val_batch = X_val_batch.to(device)
            Y_val_batch = Y_val_batch.to(device).view(-1, 1)

            val_predictions = model_bert_comparison(X_val_batch)
            val_loss = criterion_bert_comparison(val_predictions, Y_val_batch)
            val_total_loss += val_loss.item()

    avg_val_loss = val_total_loss / (len(X_val_list) / batch_size)
    comparison_bert_val_losses.append(avg_val_loss)

    print(f"BERT Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

print("BERT comparison model training complete!")

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Function took {execution_time:.6f} seconds to run.")

In [ ]:
# Plotting the training and validation loss for the BERT model
epochs = range(1, len(comparison_bert_train_losses) + 1)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Training Loss on the primary y-axis (ax1)
ax1.plot(epochs, comparison_bert_train_losses, 'bo-', label='Training loss (BERT)')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Training Loss', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a second y-axis for Validation Loss (ax2)
ax2 = ax1.twinx()
ax2.plot(epochs, comparison_bert_val_losses, 'ro-', label='Validation loss (BERT)')
ax2.set_ylabel('Validation Loss', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Add title and legend
plt.title('Training and Validation Loss (BERT)')
fig.legend(loc='upper right', bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)

plt.grid(True)
plt.savefig('training_validation_loss_bert.png')
plt.show()